# 01 — Knapsack: the Rosetta stone

**2026 Quantum Accelerator · Day 1 · 09:55 · 50 minutes**

One idea, and everything else today builds on it:

> **Every decision becomes a variable. Every rule becomes a term in one matrix.**

We use the simplest interesting problem there is. You have a set of items, each with a
value and a weight, and a bag that can only carry so much. Which items do you take?

Everyone in this room could solve the version we start with by hand. That is deliberate —
when the machinery is new, the answer should not be. By the end of the notebook you will
have written the same problem in three languages: as a list of choices, as a matrix, and
as something a solver can take.

**What you will type.** Two things: the items, and the capacity.

---
## 1. The bag

In [ ]:
import numpy as np
import itertools
import dimod
from pyqubo import Array, Constraint, Placeholder
from dwave.samplers import SimulatedAnnealingSampler
import helpers_knapsack as hk

# ------------------------------------------------------------------ CHANGE ME
ITEMS = {
    "laptop":   dict(value=8,  weight=3),
    "camera":   dict(value=47, weight=11),
    "tent":     dict(value=10, weight=14),
    "kettle":   dict(value=5,  weight=19),
    "notebook": dict(value=16, weight=5),
}
CAPACITY = 26
# ---------------------------------------------------------------------------

names = list(ITEMS)
values = np.array([ITEMS[k]["value"] for k in names])
weights = np.array([ITEMS[k]["weight"] for k in names])
n = len(names)

print(f"{'item':<12}{'value':>7}{'weight':>8}{'value/weight':>14}")
for k, v, w in zip(names, values, weights):
    print(f"{k:<12}{v:>7}{w:>8}{v / w:>14.2f}")
print(f"\ntotal weight if you take everything: {weights.sum()}   capacity: {CAPACITY}")
print(f"number of possible choices: 2^{n} = {2 ** n}")

---
## 2. Cheat first: look at every possibility

With five items there are 32 choices, so we can simply look at all of them. This is the
only time today we will be able to do that — and it is worth seeing what "the answer"
looks like before we start computing it.

In [ ]:
subsets = [s for r in range(n + 1) for s in itertools.combinations(range(n), r)]

best_value, best_set = -1, None
for s in subsets:
    w = sum(weights[i] for i in s)
    v = sum(values[i] for i in s)
    if w <= CAPACITY and v > best_value:
        best_value, best_set = v, s

EXACT_BEST = best_value
print(f"checked all {len(subsets)} choices")
print(f"best value that fits: {EXACT_BEST}")
print(f"take: {[names[i] for i in best_set]}  "
      f"(weight {sum(weights[i] for i in best_set)}/{CAPACITY})")

hk.plot_brute_force(weights, values, CAPACITY, subsets)

**Two minutes at your table.** Five items gave 32 choices. Fifty items give about 10^15 —
a thousand years at a million checks a second. A hundred items give more choices than
there are atoms in the Earth.

So enumeration is out, essentially immediately. Everything that follows is about how to
*search* instead of *look*.

---
## 3. Turning it into a matrix

A QUBO — quadratic unconstrained binary optimisation — is a problem of exactly one shape:

> choose x in {0,1}^N to minimise  x^T Q x

That is all it can express. No constraints, no inequalities, no "subject to". Just a
matrix and a vector of yes/no decisions. So we have two things to arrange.

**The objective.** We want to *maximise* value, and a QUBO *minimises*, so we minimise
`-sum(v_i x_i)`. Single-variable terms live on the diagonal of Q, because `x_i * x_i = x_i`
for binary variables.

**The capacity rule.** A QUBO cannot say "weight must be at most 26". What it can do is
*charge* you for breaking that rule. We introduce a **slack** variable `s` — the unused
space in the bag — and demand that the weight plus the slack comes to exactly the capacity:

    sum(w_i x_i) + s = CAPACITY

Then the penalty `(sum(w_i x_i) + s - CAPACITY)^2` is zero when the rule holds and positive
when it does not. The slack `s` is an integer between 0 and the capacity, and since a QUBO
only has binary variables, we write it as a sum of bits.

In [ ]:
def slack_bits(capacity):
    """Coefficients for a slack integer with exact range [0, capacity].

    Powers of two, with the last coefficient trimmed so the largest representable
    value is exactly `capacity` rather than overshooting it.
    """
    coeffs, remaining, power = [], int(capacity), 1
    while remaining > 0:
        c = min(power, remaining)
        coeffs.append(c)
        remaining -= c
        power *= 2
    return coeffs


COEFFS = slack_bits(CAPACITY)
print(f"slack coefficients: {COEFFS}")
print(f"{len(COEFFS)} bits, representing any integer from 0 to {sum(COEFFS)}")
print(f"\nso the QUBO has {n} item variables + {len(COEFFS)} slack bits "
      f"= {n + len(COEFFS)} variables")

Now build it. `LAMBDA` is how much we charge for breaking the capacity rule — set high
enough here that breaking it is never worth it. **The next notebook is entirely about
choosing this number**, so leave it alone for the moment.

In [ ]:
LAMBDA = 2.0 * float(values.sum())      # generous on purpose; see notebook 02

x = Array.create("x", shape=n, vartype="BINARY")
s = Array.create("s", shape=len(COEFFS), vartype="BINARY")

objective = -sum(int(v) * xi for v, xi in zip(values, x))
load = sum(int(w) * xi for w, xi in zip(weights, x))
slack = sum(int(c) * si for c, si in zip(COEFFS, s))
capacity_rule = Constraint((load + slack - int(CAPACITY)) ** 2, label="capacity")

model = (objective + Placeholder("lam") * capacity_rule).compile()

feed = {"lam": LAMBDA}
Q_dict, offset = model.to_qubo(feed_dict=feed)
print(f"{len(Q_dict)} non-zero entries in Q, constant offset {offset:.0f}")

### Look at it

In [ ]:
# assemble the dictionary into a dense matrix, items first then slack bits
order = [f"x[{i}]" for i in range(n)] + [f"s[{j}]" for j in range(len(COEFFS))]
pos = {name: i for i, name in enumerate(order)}
N = len(order)
Q = np.zeros((N, N))
for (a, b), coeff in Q_dict.items():
    Q[pos[a], pos[b]] += coeff

hk.show_qubo_matrix(Q, n_items=n)
print("Diagonal (one term per variable):")
print(np.round(np.diag(Q), 1))

**This is the whole idea.** Every item and every slack bit is one row and one column.
Every rule in the problem is somewhere in those numbers. The item block in the top left
and the slack block in the bottom right are coupled by the capacity rule — that is the
off-diagonal structure you can see.

**Look at the magnitudes on the diagonal, though.** The values you actually care about are
8, 47, 10, 5 and 16. The numbers in the matrix are in the tens of thousands. The capacity
*rule* outweighs the *objective* by three orders of magnitude, which is why the colour
scale on that heatmap has to be logarithmic to show anything at all.

That is not a mistake in the model. It is `LAMBDA` doing what we told it to, and it is
your first sight of the thing the next notebook is entirely about.

Nothing about this matrix knows it is a knapsack. The same shape will hold a roster this
afternoon, and a graph before lunch.

---
## 4. Solve it three ways

Exactly (possible only because the problem is tiny), by annealing, and by the
enumeration we already did.

In [ ]:
exact = dimod.ExactSolver().sample_qubo(Q_dict)
annealed = SimulatedAnnealingSampler().sample_qubo(Q_dict, num_reads=100,
                                                   num_sweeps=1000, seed=1)
decoded = model.decode_sampleset(
    SimulatedAnnealingSampler().sample(model.to_bqm(feed_dict=feed),
                                       num_reads=100, num_sweeps=1000, seed=1),
    feed_dict=feed)

legal = [d for d in decoded if not d.constraints(only_broken=True)]
best = max(legal, key=lambda d: sum(int(v) * d.array("x", i)
                                    for i, v in enumerate(values))) if legal else None

print(f"enumeration (section 2)      : {EXACT_BEST}")
print(f"exact solver on the QUBO     : {-(exact.first.energy + offset):.0f}")
if best is not None:
    picked = [names[i] for i in range(n) if best.array("x", i)]
    got = sum(int(v) * best.array("x", i) for i, v in enumerate(values))
    load_got = sum(int(w) * best.array("x", i) for i, w in enumerate(weights))
    print(f"annealing, best legal answer : {got}   {picked}  "
          f"(weight {load_got}/{CAPACITY})")
    print(f"legal in {len(legal)} of {len(decoded)} runs")
else:
    print("annealing found no legal answer -- raise LAMBDA")

All three agree. That is the point of the module: you have now expressed a decision
problem as a matrix, and a general-purpose solver that knows nothing about knapsacks found
the same answer as exhaustive search.

---
## 5. One honest warning

Someone is about to point out that knapsack is not actually hard, and they are right.
There is a classic dynamic program that solves it *exactly* in time proportional to
`items x capacity` — not `2^items`. Ten lines:

In [ ]:
def knapsack_dp(values, weights, capacity):
    """Exact, pseudo-polynomial. Solves 10,000 items faster than you can blink."""
    best = np.zeros(int(capacity) + 1, dtype=np.int64)
    for v, w in zip(values, weights):
        w = int(w)
        if w > capacity:
            continue
        best[w:] = np.maximum(best[w:], best[:-w] + int(v))
    return int(best[int(capacity)])


import time
rng = np.random.default_rng(0)
for n_big in (100, 1000, 10000):
    v_big = rng.integers(10, 100, n_big)
    w_big = rng.integers(5, 50, n_big)
    cap_big = int(0.5 * w_big.sum())
    t0 = time.perf_counter()
    opt = knapsack_dp(v_big, w_big, cap_big)
    print(f"{n_big:>6} items, capacity {cap_big:>7}: exact optimum {opt:>8}  "
          f"[{time.perf_counter() - t0:.3f}s]")

So this module is about **expression**, not about speed. We used knapsack because you
already understand it, which let the machinery be the only new thing.

Two things follow, and they are the rest of the day:

- **Where does the machinery break?** After lunch you will watch it fall apart on a
  problem where the rules pile up (13:00, nurse rostering).
- **Where does it win?** Mid-afternoon you will see the regime where this matrix form
  beats a commercial solver by orders of magnitude — dense problems with almost no rules
  at all (14:30).

The one number we glossed over, `LAMBDA`, turns out to decide both. That is the next
notebook.

---
## 6. Stretch goals

**A. Change the bag.** Make the capacity very small, or very large. At what point does the
problem become boring? Why?

**B. Count the slack bits.** Our slack used a binary expansion — about `log2(capacity)`
bits. Two alternatives: *one-hot*, a separate bit for every possible value of the slack
(so `capacity + 1` bits, with a rule that exactly one is set), and *unary*, where the
slack is the number of bits set. Work out how many variables each would need here. Which
would you pick, and does the answer depend on the capacity or on the number of items?

**C. Break it on purpose.** Set `LAMBDA = 1.0` and re-run section 4. What comes back, and
what does `best.constraints(only_broken=True)` tell you? You have just previewed notebook
02.

**D. A second rule.** Suppose the camera and the laptop cannot both be taken. Add
`Constraint(x[0] * x[1], label="camera_or_laptop")` to the model, rebuild, and re-solve.
Note that you did not have to change the objective — you added a term. That additivity is
the whole reason this form is useful.